<a href="https://colab.research.google.com/github/Abdi-dotcom/autotrader-vehicle-price-prediction/blob/main/MLC_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 . Data Understanding and Exploration

Importing and Configuring Essential Packages

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime
from scipy.stats.mstats import winsorize
from scipy import stats
from matplotlib.ticker import FuncFormatter
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import make_column_transformer
import statsmodels.api as sm
from statsmodels.formula.api import ols
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
!pip install category_encoders
from category_encoders import TargetEncoder
from sklearn import set_config

from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.datasets import load_digits
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import RidgeCV

#regression
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_selection import RFECV

sns.set(
    { "figure.figsize": (10, 6) },
    style='ticks',
    color_codes=True,
    font_scale=0.8
)
sns.set_theme(style="whitegrid")
# Improves plot display in Jupyter Notebook
%config InlineBackend.figure_formats = set(('retina', 'svg'))

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive/')


In [ ]:

file_path = '/content/drive/MyDrive/Colab Notebooks/adverts.csv'
car_dataset = pd.read_csv(file_path)


##1.1 Meaning and Type of Features; Analysis of Distributions

In [ ]:
# Gets first 5-n rows
car_dataset.head(5)

In [ ]:
# Gets last 5-n rows
car_dataset.tail(5)

In [ ]:
# The column index as property of the dataset
car_dataset.columns

In [ ]:
car_dataset.shape

We can see the dataset shows a default  "RangeIndex, starting from 0 to 402005.

In [ ]:
# This code displays the row index
car_dataset.index

The .info() methods provides a summary of the car dataset structure, showing the feature data types and the count of observations, ( for example strings/objects vs numerical). It is important for initial data cleaning, as it shows the non-null counts for each column, allowing us to identify the missing values that require imputation or removal immediately.

In [ ]:
# To get information on datatypes
car_dataset.info()

In [ ]:
# This code displays a statistical summary of numerical data like count, mean, std, min, percentiles, max
car_dataset.describe().round(2)

Identify Data Quality

In [ ]:
# "np.int64(0)" This means no duplicate rows, this is good as it means my data is clean in terms of duplication
car_dataset.duplicated().sum()

In [ ]:
# To get the missing values count per column
car_dataset.isna().sum()

In [ ]:
#display missing values percentage per column
car_dataset.isnull().sum() / len(car_dataset) * 100

In [ ]:
# This code gives us how many unique values in each column
car_dataset.nunique()

In [ ]:
car_dataset[['mileage', 'price']].describe().round(2)


In [ ]:
car_dataset['fuel_type'].value_counts(normalize=False)

In [ ]:
car_dataset.describe()

like the price, histogram and boxplot was used to identify any outliers and skewness of the mileage, the output also shows the mileage distribution is right skewed, implying the cars having lower mileage and fewer cars with very high mileage

In [ ]:
x = car_dataset['mileage'].dropna()
x_focus = x[(x >= 0) & (x <= 100_000)]

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=False)

# Histogram with KDE
sns.histplot(x_focus, bins=50, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('Distribution of Mileage (0–100,000)')
axes[0].set_xlabel('Mileage')
axes[0].set_ylabel('Frequency')

# Boxplot
sns.boxplot(x=x_focus, ax=axes[1], color='#4C72B0', showfliers=True)
axes[1].set_title('Mileage Boxplot (0–100,000)')
axes[1].set_xlabel('Mileage')
axes[1].set_ylabel('')

fig.suptitle('Mileage Distribution and Boxplot', y=1.03, fontsize=12)
plt.tight_layout()


The count plots provide insights into the composition of the car listings. Petrol and Diesel cars are the most prevalent fuel types. Used cars significantly outnumber new cars. Hatchback and SUV are the most common body types listed.

In [ ]:
# Count plots for fuel_type, vehicle_condition, and body_type
sns.countplot(y='fuel_type', data=car_dataset, order=car_dataset['fuel_type'].value_counts().index)
plt.title('Count of Cars by Fuel Type')
plt.xlabel('Count')
plt.ylabel('Fuel Type')
plt.show()

In [ ]:
sns.countplot(y='vehicle_condition', data=car_dataset, order=car_dataset['vehicle_condition'].value_counts().index)
plt.title('Count of Cars by Vehicle Condition')
plt.xlabel('Count')
plt.ylabel('Vehicle Condition')
plt.show()


In [ ]:
sns.countplot(y='body_type', data=car_dataset, order=car_dataset['body_type'].value_counts().index)
plt.title('Count of Cars by Body Type')
plt.xlabel('Count')
plt.ylabel('Body Type')
plt.show()

In [ ]:
missing_count = car_dataset.isnull().sum()
print("missing value per column:")
print(missing_count[missing_count>0].sort_values(ascending=False))



##1.2 Analysis of Predictive Power of Features





A histogram for each numerical attribute

In [ ]:
car_dataset.hist(bins=50, figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(car_dataset.corr(numeric_only=True) > 0.7, annot=True, cbar=False)
plt.show()

In [ ]:
#looking for correlations
corr_matrix = car_dataset.corr(numeric_only=True)
corr_matrix

To identify any skewness or outliers a histogram and boxplot was used. Th figures above display the price column is heavily skewed to the right. This means that most of the cars are priced lower, with few expensive outliers. The boxplot clearly suggests that presence of these outliers, and it’s recommended that further analysis is necessary.

In [ ]:

def plot_distribution(car_dataset, column):
    x = car_dataset[column].dropna().astype(float)
    p99 = np.quantile(x, 0.99)  # caps the display at 99th percentile

    plt.figure(figsize=(10, 6))
    sns.histplot(x, kde=True, color='blue', alpha=0.6, bins="fd")
    plt.axvline(x.mean(), color='red', linestyle='dashed', linewidth=2, label='Mean')
    plt.axvline(np.median(x), color='green', linestyle='dashed', linewidth=2, label='Median')
    plt.title(f'Distribution of {column} (display capped at 99th percentile)')
    plt.xlim(0, p99)
    plt.legend()
    plt.tight_layout()


In [ ]:
plot_distribution(car_dataset, 'price')

In [ ]:
plot_distribution(car_dataset, 'mileage')
#plt.savefig("mileage.png")

categorical data

In [ ]:
cat_columns = car_dataset.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
car_dataset[cat_columns].describe()

In [ ]:
print(cat_columns[1], '->', car_dataset[cat_columns[1]].mode()[0])
print(cat_columns[2], '->', car_dataset[cat_columns[2]].mode()[0])
print(cat_columns[3], '->', car_dataset[cat_columns[3]].mode()[0])
print(cat_columns[4], '->', car_dataset[cat_columns[4]].mode()[0])
print(cat_columns[5], '->', car_dataset[cat_columns[5]].mode()[0])
print(cat_columns[6], '->', car_dataset[cat_columns[6]].mode()[0])
print(cat_columns[7], '->', car_dataset[cat_columns[7]].mode()[0])

for i in cat_columns:
  formula = 'price ~ {}'.format(i)
  model = ols(formula, data=car_dataset).fit()
  annova = sm.stats.anova_lm(model, typ=2)
  p_value = annova.iloc[0,3]
  print('p-value for price ~ {}: {}'.format(i, p_value))

# 2  Data Processing for Machine Learning



##2.1 Dealing with Missing Values, Outliers, and Noise


In [ ]:
# The first part of filling missing year_of_registration based on condition
car_dataset.loc[
    (car_dataset['mileage'].between(0, 2000, inclusive='both'))&
    (car_dataset['reg_code'].isna() | (car_dataset['reg_code'].astype(str).str.strip()==''))&
    (car_dataset['vehicle_condition'] == 'NEW')&
    (car_dataset['year_of_registration'].isna()), 'year_of_registration'

] = 2020

In [ ]:
# Defined the DVLA registration code to year mapping
reg_to_year = {
    '51': 2001, '02': 2002, '52': 2002,
    '03': 2003, '53': 2003,
    '04': 2004, '54': 2004,
    '05': 2005, '55': 2005,
    '06': 2006, '56': 2006,
    '07': 2007, '57': 2007,
    '08': 2008, '58': 2008,
    '09': 2009, '59': 2009,
    '10': 2010, '60': 2010,
    '11': 2011, '61': 2011,
    '12': 2012, '62': 2012,
    '13': 2013, '63': 2013,
    '14': 2014, '64': 2014,
    '15': 2015, '65': 2015,
    '16': 2016, '66': 2016,
    '17': 2017, '67': 2017,
    '18': 2018, '68': 2018,
    '19': 2019, '69': 2019,
    '20': 2020, '70': 2020,
    '21': 2021, '71': 2021,
    '22': 2022, '72': 2022,
    '23': 2023, '73': 2023,
    '24': 2024, '74': 2024,
    '25': 2025, '75': 2025
}

# Cleaned reg_code column and convert to string
car_dataset['year_of_registration'] = car_dataset['year_of_registration'].astype('Int64')
car_dataset['reg_code'] = car_dataset['reg_code'].astype(str).str.strip()

# Explicitly corrected the  reg_code issue
car_dataset.loc[car_dataset['reg_code'] == '58', 'year_of_registration'] = 2008
car_dataset.loc[car_dataset['reg_code'] == '50', 'year_of_registration'] = 2000


# Filling the missing year_of_registration where possible based on reg_code
car_dataset.loc[
    (car_dataset['year_of_registration'].isna()) &
    (car_dataset['reg_code'].isin(reg_to_year.keys())),
    'year_of_registration'
] = car_dataset.loc[
    (car_dataset['year_of_registration'].isna()) &
    (car_dataset['reg_code'].isin(reg_to_year.keys())),
    'reg_code'
].map(reg_to_year)


car_dataset.loc[
    car_dataset['reg_code'].isin(reg_to_year.keys()),
    'year_of_registration'
] = car_dataset.loc[
    car_dataset['reg_code'].isin(reg_to_year.keys()),
    'reg_code'
].map(reg_to_year)

nan_reg_code_mask = car_dataset['reg_code'] == 'nan'

car_dataset.loc[nan_reg_code_mask, 'reg_code'] = car_dataset.loc[nan_reg_code_mask, 'year_of_registration'].astype(str).str[-2:]

avg_reg_year = int(car_dataset[car_dataset['vehicle_condition'] == 'USED']['year_of_registration'].mean())

car_dataset.loc[(car_dataset['vehicle_condition'] == 'USED') & (car_dataset['year_of_registration'].isna()), 'year_of_registration'] = avg_reg_year

rows_999_year = car_dataset['year_of_registration'] == 999

car_dataset.loc[
    rows_999_year & car_dataset['reg_code'].isin(reg_to_year.keys()),
    'year_of_registration'
] = car_dataset.loc[
    rows_999_year & car_dataset['reg_code'].isin(reg_to_year.keys()),
    'reg_code'
].map(reg_to_year)


car_dataset.loc[car_dataset['year_of_registration'] == 999, 'year_of_registration'] = avg_reg_year

In [ ]:
car_dataset['year_of_registration'] = car_dataset['year_of_registration'].replace(2023, 2020)

In [ ]:
# Converted reg_code to string type
car_dataset['reg_code'] = car_dataset['reg_code'].astype(str)

# Replaced '.0' at the end of strings if present
car_dataset['reg_code'] = car_dataset['reg_code'].str.replace(r'\.0$', '', regex=True)

#car_dataset['reg_code'].value_counts().sort_index().head(30)
car_dataset['reg_code'].info()

In [ ]:
# Calculated the mean year of registration for used cars and convert it to an integer
avg_reg_year = int(car_dataset[car_dataset['vehicle_condition'] == 'USED']['year_of_registration'].mean())

# Filled missing year_of_registration values for used cars with the calculated mean
car_dataset.loc[(car_dataset['vehicle_condition'] == 'USED') & (car_dataset['year_of_registration'].isna()), 'year_of_registration'] = avg_reg_year


Cleaning and filling the missing values of Standard_colour

In [ ]:

# I group the standard make of the car and get the avarage of each brand colour and replaced the empty values it
make_colour_mode = car_dataset.groupby('standard_make')['standard_colour'].agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
make_colour_mode = make_colour_mode.to_dict()
car_dataset['standard_colour'] = car_dataset.apply(lambda row: make_colour_mode.get(row['standard_make'])
if pd.isnull(row['standard_colour']) else row['standard_colour'], axis= 1 )


missing_colour= car_dataset[car_dataset['standard_colour'].isna()]

# Calculated the overall most frequent colour
most_frequent_colour = car_dataset['standard_colour'].mode()[0]

# Replaced the missing 'standard_colour' values with the most frequent colour
car_dataset['standard_colour'].fillna(most_frequent_colour, inplace=True)

I have imputed the missing mileage values in the dataset by using grouped considering features that are likely to influence mileage, such as standard_make, standard_model, and year_of_registration. and calculate the median mileage for the entire dataset as a fallback.

In [ ]:
grouped_mileage = car_dataset.groupby(['standard_make', 'standard_model', 'year_of_registration'])['mileage'].median()

overall_median_mileage = car_dataset['mileage'].median()

car_dataset['mileage'] = car_dataset.apply(
    lambda row: grouped_mileage.get((row['standard_make'], row['standard_model'], row['year_of_registration']), overall_median_mileage)
    if pd.isnull(row['mileage']) else row['mileage'], axis=1
)
car_dataset['mileage'].fillna(overall_median_mileage, inplace=True)

In [ ]:
# this code finds the most frequent value(mode) in the body type and fuel type and
#replaces all the missing values in place with the mode
body_type_mode = car_dataset['body_type'].mode()[0]
car_dataset['body_type'].fillna(body_type_mode, inplace=True)
# for fuel type
fuel_type_mode = car_dataset['fuel_type'].mode()[0]
car_dataset['fuel_type'].fillna(fuel_type_mode, inplace=True)

In [ ]:
car_dataset.isnull().sum()

In [ ]:
car_dataset['price'].info()


Cross-Feature Validation, to check if combination make sense(uel_type = Electric with body_type = SUV is valid, but fuel_type = Electric with year_of_registration = 1990 might be suspicious)

In [ ]:
invalid_rows = car_dataset[(car_dataset['fuel_type'] == 'Electric') & (car_dataset['year_of_registration'] < 2000)]
invalid_rows

In [ ]:

for col in ['standard_make', 'standard_model', 'standard_colour', 'body_type', 'fuel_type']:
    freq = car_dataset[col].value_counts(normalize=True)
    rare_values = freq[freq < 0.01].index.tolist()
    print(f"Rare categories in {col}: {rare_values}")


In [ ]:
car_dataset.groupby('standard_make')['price'].describe()

**Handling Outliers**

using winsorization and stratificion to handle the extrame outliers in price

In [ ]:
def stratified_winsorize(group):
  group['price_winsorized'] = winsorize(group['price'],
                              limits=[0.01, 0.01])
  return group

car_dataset = car_dataset.groupby('vehicle_condition', group_keys=False).apply(stratified_winsorize)

car_dataset[['price', 'price_winsorized']].describe().round(1)

In [ ]:


fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(
    x=car_dataset['price'],
    ax=ax[0],

    fliersize=3,
    color='#1f77b4'
)
ax[0].set_title('Boxplot of Price (with outliers)', fontsize=14)
ax[0].set_xlabel('Price (£)', fontsize=12)


ax[0].set_xlim(car_dataset['price'].quantile(0.01), car_dataset['price'].quantile(0.95) * 1.5)



sns.boxplot(
    x=car_dataset['price_winsorized'],
    ax=ax[1],

    color='#ff7f0e'
)
ax[1].set_title('Boxplot of Price_Winsorized (without outliers)', fontsize=14)
ax[1].set_xlabel('Price (Winsorized £)', fontsize=12)


ax[1].set_xlim(car_dataset['price_winsorized'].min(), car_dataset['price_winsorized'].max() * 1.1)

plt.tight_layout()
plt.show()

In [ ]:


def stratified_winsorize(group):
  group['mileage_winsorized'] = winsorize(group['mileage'], limits=[0.01, 0.01])
  return group

car_dataset = car_dataset.groupby('vehicle_condition', group_keys=False).apply(stratified_winsorize)

car_dataset[['mileage', 'mileage_winsorized']].describe().round(2)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(
    x=car_dataset['mileage'],
    ax=ax[0],
    fliersize=3,
    color='#1f77b4'
)
ax[0].set_title('Boxplot of mileage (with outliers)', fontsize=14)
ax[0].set_xlabel('Mileage (Miles)', fontsize=12)


ax[0].set_xlim(car_dataset['mileage'].quantile(0.01), car_dataset['mileage'].quantile(0.99) * 1.5)


sns.boxplot(
    x=car_dataset['mileage_winsorized'],
    ax=ax[1],
    color='#ff7f0e' )

ax[1].set_title('Boxplot of Mileage Winsorized (without outliers)', fontsize=14)
ax[1].set_xlabel('Mileage (Winsorized Miles)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
car_dataset['price'].describe().round(2)

In [ ]:
car_dataset['mileage_winsorized'].max()

In [ ]:
car_dataset.info()

In [ ]:
car_dataset.groupby('standard_make')['price_winsorized'].describe()

In [ ]:
z_scores = np.abs(stats.zscore(car_dataset['price']))
outliers= np.where(z_scores > 3)
outliers

In [ ]:
# Converts 'year_of_registration' to integer type
car_dataset['year_of_registration'] = car_dataset['year_of_registration'].astype(int)


In [ ]:
recent_years_data = car_dataset[(car_dataset['year_of_registration'] >= 2000) & (car_dataset['year_of_registration'].astype(int) <= 2020)]

# Generates histogram for year_of_registration
sns.histplot(data=recent_years_data, x='year_of_registration', bins=20, kde=True)
plt.title('Distribution of Year of Registration (2000-2020)')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.xticks(range(2000, 2021, 2)) # Sets x-axis ticks to show integer years
plt.show()

print("\nValue counts for year_of_registration (2000-2020):")
print(recent_years_data['year_of_registration'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=car_dataset, x='price_winsorized', bins=10, kde=True, color='Darkblue')
plt.title('Distribution of Price (Winsorized)')
plt.xlabel('price')
plt.ylabel('Frequency')

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(data=car_dataset, x='mileage_winsorized', bins=10, kde=True, color='darkblue')
plt.title('Distribution of Mileage (Winsorized)')
plt.xlabel('Mileage')
plt.ylabel('Frequency')

In [ ]:

summary = car_dataset.groupby(['standard_make', 'fuel_type', 'body_type'])['price_winsorized'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)
print("\nTop Combinations by Average Price:\n", summary.head(15))


In [ ]:
# Checks for empty values in the 'year_of_registration' column
missing_years = car_dataset['year_of_registration'].isnull().sum()

print(f"Number of empty values in 'year_of_registration': {missing_years}")

In [ ]:
# Filters rows where 'reg_code' contains only alphabetic characters
char_reg_codes = car_dataset[car_dataset['reg_code'].str.isalpha()]

char_reg_code_counts = char_reg_codes['reg_code'].value_counts()

print("Count of character-based reg_codes:")
print(char_reg_code_counts)

In [ ]:
car_dataset['year_of_registration'].isnull().sum()

In [ ]:
# Filters data for vehicles registered in 2000 or later
recent_years_data = car_dataset[car_dataset['year_of_registration'] >= 2000]

sns.histplot(data=recent_years_data, x='year_of_registration', bins=20, kde=True)
plt.title('Distribution of Year of Registration (2000 onwards)')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Calculates the percentage of each vehicle condition
condition_counts = car_dataset['vehicle_condition'].value_counts(normalize=True) * 100

plt.figure(figsize=(8, 8))
plt.pie(condition_counts, labels=condition_counts.index, autopct='%1.1f%%', startangle=90, colors=['skyblue', 'lightcoral'])
plt.title('Percentage of Vehicle Condition (New vs Used)')
plt.show()

In [ ]:
display(car_dataset['body_type'].value_counts(normalize=True) * 100)

In [ ]:
car_dataset.loc[car_dataset['year_of_registration'] < 2000, ["standard_make", "standard_model", "reg_code", "year_of_registration"]]

In [ ]:
#numeric sanity check
numeric_columns = car_dataset.select_dtypes(include=['int64', 'float64']).columns.tolist()
car_dataset[numeric_columns].describe()

In [ ]:
# categorical sanity check
cat_columns = car_dataset.select_dtypes(exclude=['int64', 'float64']).columns.tolist()
car_dataset[cat_columns].describe()

In [ ]:
def cat_checker(catIndex):
  values = np.sort(car_dataset[cat_columns[catIndex]].unique())
  return cat_columns[catIndex], values


In [ ]:
  cat_checker(6)

##2.2 Feature Engineering, Data Transformations, Feature Selection



####Feature Engineering

In [ ]:
car_dataset.drop(['public_reference','reg_code'], axis=1, inplace=True)

In [ ]:
current_year = 2025
car_dataset['vehicle_age'] = current_year - car_dataset['year_of_registration']
car_dataset['log_mileage'] = np.log(car_dataset['mileage_winsorized'] + 1)
car_dataset['price_per_mile'] = car_dataset['price_winsorized'] / (car_dataset['mileage_winsorized'] + 1)
car_dataset['condition_encoded'] = car_dataset['vehicle_condition'].map({'NEW': 1, 'USED': 0})
car_dataset['crossover_flag'] = car_dataset['crossover_car_and_van'].astype(int)
car_dataset['mileage_per_year'] = car_dataset['mileage_winsorized'] / car_dataset['vehicle_age']


In [ ]:

model_counts = car_dataset['standard_model'].value_counts()
car_dataset['Model_Count'] = car_dataset['standard_model'].map(model_counts)


car_dataset['Age_x_Mileage'] = car_dataset['vehicle_age'] * car_dataset['log_mileage']

In [ ]:
car_dataset['Model_Count']

Original vs log mileage distribution plot

In [ ]:

# Creates figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
# Original mileage mistribution
sns.histplot(car_dataset['mileage_winsorized'], bins=40, color='skyblue', kde=True, ax=axes[0])
axes[0].set_title('Original mileage distribution')
axes[0].set_xlabel('Mileage')
axes[0].set_ylabel('Frequency')
# Log mileage distribution
sns.histplot(car_dataset['log_mileage'], bins=40, color='salmon', kde=True, ax=axes[1])
axes[1].set_title('Log transformed mileage distribution')
axes[1].set_xlabel('Log(Mileage)')
axes[1].set_ylabel('Frequency')
plt.suptitle('Original vs Log Mileage', fontsize=12)
plt.tight_layout()
#plt.savefig("original vs log mileage.png")
plt.show()



 Weighted Model Popularity Feature Engineering

In [ ]:
# calculated frequancy of each model demand
model_freq = car_dataset['standard_model'].value_counts()
# calcaluated avarage price per model
model_price_mean = car_dataset.groupby('standard_model')['price_winsorized'].mean()

# to improve the code , its normalised both components by scaling 0-1 scale
model_freq_norm = (model_freq - model_freq.min()) / (model_freq.max() - model_freq.min())
price_norm = (model_price_mean - model_price_mean.min()) / (model_price_mean.max() - model_price_mean.min())

weighted_score = (0.6 * model_freq_norm) + (0.4 * price_norm)
car_dataset['model_popularity'] = car_dataset['standard_model'].map(weighted_score)


top_models = car_dataset[['standard_model', 'model_popularity']].drop_duplicates().sort_values('model_popularity', ascending=False)
print("Top Models by Weighted Popularity:\n", top_models.head(10))


# Createted classes based on quantiles
car_dataset['popularity_class'] = pd.qcut(car_dataset['model_popularity'], q=3, labels=['Low', 'Medium', 'High'])

# to Check distribution of classes
print(car_dataset['popularity_class'].value_counts())

top_models['popularity_class'] = top_models['model_popularity'].apply(
    lambda x: 'High' if x > car_dataset['model_popularity'].quantile(0.66)
    else 'Medium' if x > car_dataset['model_popularity'].quantile(0.33)
    else 'Low'
)
print(top_models.head(10))

In [ ]:
print(car_dataset['popularity_class'].value_counts())

In [ ]:
car_dataset[car_dataset['popularity_class']== 'High']

Budget-Friendly (Under £18,000),Mid-Range (£18,000 to £50,000),Luxury/High-Performance (£55,000+)

In [ ]:
bins = [0, 18000,50000, float(np.inf)]

labels = ['Budget Friendly', 'Mid-Range', 'luxury/High-Performance']

car_dataset['price_bucket'] = pd.cut(car_dataset['price_winsorized'], bins=bins, labels=labels, include_lowest=True)

In [ ]:
print(car_dataset['price_bucket'].value_counts())

In [ ]:

bucket_counts = car_dataset['price_bucket'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(bucket_counts, labels=bucket_counts.index, autopct='%1.1f%%', colors=['#66b3ff','#99ff99','#ff9999'])
plt.title('Proportion of Price Buckets')
#plt.savefig("Proportion of Price Buckets")
plt.show()


In [ ]:
print(car_dataset['fuel_type'].value_counts())

The  the fuel categoies  ware unbalanced and so new categories ware created. patrol and diesel remains the same as they are major categories, all the hybrid variant(Petrol Hybrid, Petrol Plug-in Hybrid, Diesel Hybrid, Diesel Plug-in Hybrid) are grouped under Hybrid.Electric is distinct and kept separate. andBi Fuel and Natural Gas are rare and can be grouped under Other to reduce sparsity.

In [ ]:
# defining the mapping for fuel type
fuel_type_mapping = {
    'Petrol': 'Petrol',
    'Diesel': 'Diesel',
    'Petrol Hybrid': 'Hybrid',
    'Petrol Plug-in Hybrid': 'Hybrid',
    'Diesel Hybrid': 'Hybrid',
    'Diesel Plug-in Hybrid': 'Hybrid',
    'Electric': 'Electric',
    'Bi Fuel': 'Other',
    'Natural Gas': 'Other'}

  # applying the new mapping
car_dataset['New_fuel_type']= car_dataset['fuel_type'].map(fuel_type_mapping)

In [ ]:
print(car_dataset['New_fuel_type'].value_counts())

In [ ]:
# Sets up the figure with two subplots side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

original_order = car_dataset['fuel_type'].value_counts().index
new_order = car_dataset['New_fuel_type'].value_counts().index

# original fuel_type distribution
sns.countplot(x='fuel_type', data=car_dataset, ax=axes[0], color='Blue', order=original_order)
axes[0].set_title('Original Fuel Type Distribution')
axes[0].set_xlabel('Count')
axes[0].set_ylabel('Fuel Type')
axes[0].tick_params(axis='x', rotation=45)

# new fuel_type distribution
sns.countplot(x='New_fuel_type', data=car_dataset, ax=axes[1], color='Green', order=new_order)
axes[1].set_title('Mapped Fuel Type Distribution')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('New Fuel Type')
plt.tight_layout()
axes[0].tick_params(axis='x', rotation=45)
#plt.savefig('side_by_side_fuel_distribution.png')

print("Side-by-side distribution chart saved as 'side_by_side_fuel_distribution.png'.")

Year of Registration vs Price Bucket

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=car_dataset, x='price_bucket', y='year_of_registration', palette='Set3')
plt.title('Year of Registration by Price Bucket')
plt.xlabel('Price Bucket')
plt.ylabel('Year of Registration')
plt.ylim(2000, 2023)
plt.yticks(range(2000, 2021, 2))

medians = car_dataset.groupby('price_bucket')['year_of_registration'].median()
for i, median in enumerate(medians):
    plt.text(i, median + 0.3, f'Median: {int(median)}', ha='center', color='black', fontsize=10)

plt.text(2, 2018, 'Luxury cars skew newer', color='darkblue', fontsize=10)
plt.text(0, 2004, 'Budget cars cluster older', color='darkblue', fontsize=10)
plt.tight_layout()
#plt.savefig("Year of Registration by Price Bucket.png")
plt.show()

In [ ]:
car_dataset.hist(bins=50, figsize=(10, 8))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.heatmap(car_dataset.corr(numeric_only=True) > 0.7, annot=True, cbar=False)
plt.show()

In [ ]:
#looking for correlations
corr_matrix = car_dataset.corr(numeric_only=True)
corr_matrix

In [ ]:
correlations_with_price = corr_matrix['price_winsorized'].abs().sort_values(ascending=False)
# This will visually sort the heatmap, placing features most correlated with price_winsorized at the top/left
sorted_corr_matrix = corr_matrix.loc[correlations_with_price.index, correlations_with_price.index]

plt.figure(figsize=(14, 12))
sns.heatmap(sorted_corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5, cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Sorted Correlation Matrix (by absolute correlation with price_winsorized)', fontsize=16)
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()


In [ ]:

TARGET_COL = 'price_winsorized'
TOP_N      = 9
EXCLUDE_LEAKAGE = [
    'price',
    'price_per_mile'
]

assert TARGET_COL in car_dataset.columns, f"Target '{TARGET_COL}' not found in DataFrame."

num_df = car_dataset.select_dtypes(include='number').copy()

corr_series = num_df.corr(method='spearman')[TARGET_COL].drop(TARGET_COL, errors='ignore')

toto_drop = [c for c in EXCLUDE_LEAKAGE if c in corr_series.index]
corr_series = corr_series.drop(labels=toto_drop, errors='ignore')


corr_top = corr_series.reindex(corr_series.abs().sort_values(ascending=False).head(TOP_N).index)


sns.set(style='whitegrid', context='notebook')
plt.figure(figsize=(6.5, 0.52 * len(corr_top) + 2))
ax = sns.heatmap(
    corr_top.to_frame(name=f"Spearman ρ with {TARGET_COL}"),
    annot=True, fmt=".2f", cmap='coolwarm', center=0, cbar=False
)
plt.title(f"Top {len(corr_top)} Numeric Features by Spearman ρ with {TARGET_COL}", fontsize=13)
plt.ylabel("Feature")
plt.tight_layout()
plt.show()
#plt.savefig("Correlation with Target.png")


In [ ]:


# Simple scatter + LOWESS trend for Vehicle Age
sns.set(style='whitegrid', context='notebook')
plt.figure(figsize=(7, 5))

sns.regplot(
    data=car_dataset, x='vehicle_age', y='price_winsorized',
    lowess=True,
    scatter_kws={'alpha': 0.3, 's': 12},
    line_kws={'color': 'red', 'lw': 2}
)

plt.title("Price vs Vehicle Age")
plt.xlabel("Vehicle Age (Years)")
plt.ylabel("Price")
plt.tight_layout()


In [ ]:
# Sample for faster plotting
sample_df = car_dataset.sample(40000, random_state=42)

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Body Type Boxplot (no ordering for speed)
sns.boxplot(data=sample_df, x='body_type', y='price_winsorized', ax=axes[0], showfliers=False)
axes[0].set_title('Body Type vs Price')
axes[0].tick_params(axis='x', rotation=30)

# Fuel Type Violin
sns.violinplot(data=sample_df, x='New_fuel_type', y='price_winsorized', ax=axes[1], inner="quartile", cut=0)
axes[1].set_title('Fuel Type vs Price')

fig.suptitle('Price Distribution by Body Type and Fuel Type', y=1.02, fontsize=14)
plt.tight_layout()
plt




percenatge share of brand, standared make vs percentage share

In [ ]:

brand_counts = car_dataset['standard_make'].value_counts(normalize=True) * 100
brand_df = brand_counts.reset_index()
brand_df.columns = ['Brand', 'Percentage Share']

# Sorts and selects top 10
brand_df = brand_df.sort_values(by='Percentage Share', ascending=False).head(10)

plt.figure(figsize=(11, 6))
sns.barplot(x='Brand', y='Percentage Share', data=brand_df, palette='viridis')
plt.title('Top 10 Car Brands by Percentage Share')
plt.ylabel('Percentage (%)')
plt.xlabel('Standard Make')


for index, row in brand_df.reset_index().iterrows():
    plt.text(index, row['Percentage Share'] + 0.05, f"{row['Percentage Share']:.2f}%",
             ha='center', va='bottom', fontsize=10, color='black')
#plt.savefig("Top 10 Car Brands by Percentage Share.png")
plt.tight_layout()
plt.show()



In [ ]:
unique_manufacturers = car_dataset['standard_make'].nunique()
print(f"The dataset contains a total of {unique_manufacturers} unique manufacturers.")

price  and fuel boxplot, selling price by fuel type

####Data Transformation/categorical Encoding

In [ ]:
# Calculates the mean and median for annotation
mean_price = car_dataset['price_winsorized'].mean()
median_price = car_dataset['price_winsorized'].median()

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

sns.histplot(
    car_dataset['price_winsorized'],
    bins=75, # Increase bins for smoother visualization
    kde=True,
    color='skyblue',
    edgecolor='gray',
    line_kws={'linewidth': 3}
)

# Adds Mean Line and Label
plt.axvline(
    mean_price,
    color='red',
    linestyle='dashed',
    linewidth=3,
    label=f'Mean Price: £{mean_price:,.0f}'
)
plt.text(
    mean_price * 1.05,
    plt.gca().get_ylim()[1] * 0.9, # Places text near the top
    f'Mean\n£{mean_price:,.0f}',
    color='red',
    fontsize=11,
    ha='left'
)

plt.axvline(
    median_price,
    color='green',
    linestyle='dashed',
    linewidth=3,
    label=f'Median Price: £{median_price:,.0f}'
)
plt.text(
    median_price * 0.95,
    plt.gca().get_ylim()[1] * 0.9,
    f'Median\n£{median_price:,.0f}',
    color='green',
    fontsize=11,
    ha='right'
)

plt.title(
    'Distribution of Vehicle Price (Winsorized) - Highly Skewed',
    fontsize=16,
    fontweight='bold',
    pad=20
)
plt.xlabel('Price (Winsorized)', fontsize=14)
plt.ylabel('Count of Vehicles', fontsize=14)
plt.ticklabel_format(style='plain', axis='x')
plt.grid(axis='y', linestyle='--', alpha=0.6)
#plt.savefig("Distribution of Vehicle Price (Winsorized.png")
plt.show()

In [ ]:
car_dataset.columns

In [ ]:
car_dataset[["price", "price_winsorized"]].describe().round(2)

In [ ]:
car_dataset[
    (car_dataset['vehicle_age']<0)|
    (car_dataset['mileage_winsorized']<0)
]

Mean-target endcoding:  brand and model have mnay unique value high cardinality,using mean target encoding. calculating the mean target price for each category, replace category in with its corrponding mean price, or use Ordinal Encoding which Maps each category to a unique integer value while preserving the intrinsic rank or order of the categories.

Features Suitable for Ordinal Encoding


popularity_class

Categories: Low, Medium, High
Natural order: Low < Medium < High → Perfect for ordinal encoding.



price_bucket

Categories: Budget-Friendly, Mid-Range, Luxury/High-Performance
Natural order: Budget < Mid < Luxury → Ordinal encoding makes sense.

In [ ]:
car_dataset.columns

In [ ]:

# Columns to drop from X
leakage_or_redundant = [
    'price', 'price_winsorized',       # targets
    'price_bucket', 'price_per_mile',  # derived from target
    'mileage_winsorized',              # duplicate of mileage
    'year_of_registration',            # redundant with vehicle_age
    'fuel_type',                       # prefer New_fuel_type
    'crossover_car_and_van',           # prefer crossover_flag
    'vehicle_condition',               # prefer condition_encoded
    'standard_colour'
]


In [ ]:
X = car_dataset.drop(columns= leakage_or_redundant)
y = car_dataset['price_winsorized']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:

# Identify numeric and categorical on the original data
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()


In [ ]:

# Categorical features
categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()


In [ ]:
popularity_order = ['Low', 'Medium', 'High']

ordinal_cols = ["popularity_class"]

ordinal_encoder = OrdinalEncoder(
    categories=[popularity_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [ ]:
te_cols = ["standard_make", "standard_model"]
ohe_cols = ["body_type", "New_fuel_type"]


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])


In [ ]:
numeric_transformer

In [ ]:

# Target encoder pipeline for TE columns
te_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('te', TargetEncoder(cols=None))
])


In [ ]:
te_transformer

In [ ]:

ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ord', ordinal_encoder)
])


In [ ]:
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='if_binary'))
])


In [ ]:
categorical_transformer

Pre-Processing

In [ ]:

# Combines into a single ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('te', te_transformer, te_cols),
        ('ord', ordinal_transformer, ordinal_cols),
        ('cat', categorical_transformer, ohe_cols),
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)


In [ ]:

# Fit on training only; transform train and test
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed  = preprocessor.transform(X_test)


In [ ]:
print(f"Train shape: {X_train_processed.shape}, Test shape: {X_test_processed.shape}")

In [ ]:
preprocessor

In [ ]:
try:
    feature_names = preprocessor.get_feature_names_out()
except Exception:
    # Fallback for older sklearn versions or if verbose_feature_names_out is False
    feature_names = [f"f_{i}" for i in range(X_train_processed.shape[1])]

Xtrain = pd.DataFrame(X_train_processed, columns=feature_names)
Xtest = pd.DataFrame(X_test_processed,  columns=feature_names)

# Quick sanity check (one-liner)
print(Xtrain.apply(lambda s: pd.api.types.is_numeric_dtype(s)).all() and Xtrain.notna().all().all())

In [ ]:
print(f"Train shape: {Xtrain.shape} | Test shape: {Xtrain.shape}")
Xtrain.head()


In [ ]:
y_train.value_counts()

# 3  Model Building



##3.1 . Algorithm Selection, Model Instantiation and Configuration


##3.2 Grid Search, Model Ranking and Selection


Goal: Predict selling price using car features.
Regression problem → continuous target variable.

Model fitting: with feature selction and without feature selection (FS)

Linear Regression

In [ ]:
regr_noFS = Pipeline(
    steps=[
    ("preprocessor", preprocessor),
    ('regr', LinearRegression())
])

In [ ]:

regr_noFS['preprocessor'].set_output(transform='pandas')

X_train_processed = regr_noFS['preprocessor'].fit_transform(X_train, y_train)
print(X_train_processed.columns.tolist())


In [ ]:
regr_noFS.fit(X_train, y_train)

In [ ]:
regr_noFS.score(X_test, y_test), regr_noFS.score(X_train, y_train)

In [ ]:
mean_absolute_error(y_test, regr_noFS.predict(X_test)), mean_absolute_error(y_train, regr_noFS.predict(X_train))

In [ ]:
eval_results = cross_validate(
    regr_noFS, X_train, y_train, cv=5,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)
(
    (-eval_results["test_neg_mean_absolute_error"].mean()).round(2), eval_results["test_neg_mean_absolute_error"].std(),
     (-eval_results["train_neg_mean_absolute_error"].mean()).round(2), eval_results["train_neg_mean_absolute_error"].std()
)


In [ ]:
regr_noFS['preprocessor'].get_feature_names_out()

In [ ]:
len(regr_noFS['preprocessor'].get_feature_names_out())

With Best-k

In [ ]:
regr_bestk = Pipeline(
    steps=[
    ("preprocessor", preprocessor),
    ("featsel", SelectKBest(f_regression, k=10)),
    ('regr', LinearRegression())
])

In [ ]:
regr_bestk['preprocessor'].set_output(transform='pandas')

In [ ]:
regr_bestk.fit(X_train, y_train)


In [ ]:
regr_bestk['featsel'].get_feature_names_out()

In [ ]:
len(regr_bestk['featsel'].get_feature_names_out())


In [ ]:
pd.DataFrame(regr_bestk['featsel'].transform(
    regr_bestk['preprocessor'].transform(X_train)
)).head()

In [ ]:
regr_bestk.score(X_test, y_test)

In [ ]:
mean_absolute_error(y_test, regr_bestk.predict(X_test)), mean_absolute_error(y_train, regr_bestk.predict(X_train))

In [ ]:
eval_results = cross_validate(
    regr_bestk, X_train, y_train, cv=5,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)
(
    (-eval_results["test_neg_mean_absolute_error"].mean()).round(), eval_results["test_neg_mean_absolute_error"].std(),
    -eval_results["train_neg_mean_absolute_error"].mean(), eval_results["train_neg_mean_absolute_error"].std()
)

Linear Regression

In [ ]:
model = LinearRegression()

regr_rfe = Pipeline(
    steps=[
    ("preprocessor", preprocessor),
    ("featsel", RFECV(model, step=1, cv=5)),
    ('regr', LinearRegression())]
)


In [ ]:
regr_rfe.fit(X_train, y_train)

In [ ]:
regr_rfe['featsel'].get_feature_names_out()

In [ ]:
len(regr_rfe['featsel'].get_feature_names_out())

In [ ]:
regr_rfe.score(X_test, y_test), regr_rfe.score(X_train, y_train)

In [ ]:
mean_absolute_error(y_test, regr_rfe.predict(X_test)), mean_absolute_error(y_train, regr_rfe.predict(X_train))

In [ ]:
eval_results =  cross_validate(
    regr_rfe, X_train, y_train, cv=5,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)

(
 (-eval_results["test_neg_mean_absolute_error"].mean()).round(), eval_results["test_neg_mean_absolute_error"].std(),
  -eval_results["train_neg_mean_absolute_error"].mean(), eval_results["train_neg_mean_absolute_error"].std()
)

kNN regressor

kNN pipeline (without feature selection)

Fit preprocessor on full training data

In [ ]:
preprocessor.set_output(transform="pandas")

X_train_proc = preprocessor.fit_transform(X_train, y_train)
X_test_proc  = preprocessor.transform(X_test)


Stratified subsample AFTER preprocessing

In [ ]:
y_bins = pd.qcut(y_train, q=10, duplicates="drop")

X_train_knn, _, y_train_knn, _ = train_test_split(
    X_train_proc,
    y_train,
    train_size=0.2,
    random_state=42,
    stratify=y_bins
)

Train kNN on subsampled, preprocessed data

In [ ]:
knn_reg = KNeighborsRegressor(
    n_neighbors=11,
    weights="distance",
    algorithm="ball_tree"
)

knn_reg.fit(X_train_knn, y_train_knn)


In [ ]:
y_pred_knn = knn_reg.predict(X_test_proc)

r2_score(y_test, y_pred_knn), mean_absolute_error(y_test, y_pred_knn)


In [ ]:
cross_validate(
    knn_reg,
    X_train_knn,
    y_train_knn,
    cv=3,
    scoring="neg_mean_absolute_error"
)


Decision Tree pipeline (without scaling)

In [ ]:
tree_reg = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regr", DecisionTreeRegressor(random_state=42))
    ]
)

In [ ]:
tree_reg.fit(X_train, y_train)

In [ ]:
tree_reg.score(X_test, y_test), tree_reg.score(X_train, y_train)

In [ ]:
# Training vs Test performance
tree_reg.score(X_test, y_test), tree_reg.score(X_train, y_train)

# Error metrics
mean_absolute_error(y_test, tree_reg.predict(X_test)), \
mean_absolute_error(y_train, tree_reg.predict(X_train))

Cross-validated evaluation

In [ ]:
eval_results_tree = cross_validate(
    tree_reg,
    X_train,
    y_train,
    cv=3,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)
(
    (-eval_results_tree["test_neg_mean_absolute_error"].mean()).round(2),
     eval_results_tree["test_neg_mean_absolute_error"].std(),
    (-eval_results_tree["train_neg_mean_absolute_error"].mean()).round(2),
     eval_results_tree["train_neg_mean_absolute_error"].std()
)

Tuning the descision tree

In [ ]:
param_grid_tree = {
    "regr__max_depth": [5, 10, 15, 20],
    "regr__min_samples_leaf": [5, 10, 20, 50],
    "regr__min_samples_split": [10, 20, 50]
}

tree_search = GridSearchCV(
    estimator=tree_reg,
    param_grid=param_grid_tree,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

print("Tuning Decision Tree...")

tree_search.fit(X_train, y_train)

best_tree = tree_search.best_estimator_

print("Best Decision Tree parameters:")
print(tree_search.best_params_)


Tuned tree evaluation

In [ ]:
# Test performance
best_tree.score(X_test, y_test), best_tree.score(X_train, y_train)

mean_absolute_error(y_test, best_tree.predict(X_test)), \
mean_absolute_error(y_train, best_tree.predict(X_train))

# Cross-validation
eval_results_tree_tuned = cross_validate(
    best_tree,
    X_train,
    y_train,
    cv=3,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)

(
    (-eval_results_tree_tuned["test_neg_mean_absolute_error"].mean()).round(2),
     eval_results_tree_tuned["test_neg_mean_absolute_error"].std(),
    (-eval_results_tree_tuned["train_neg_mean_absolute_error"].mean()).round(2),
     eval_results_tree_tuned["train_neg_mean_absolute_error"].std()
)


Evaluating the tuned tree on test and train sets

In [ ]:
# Predictions
y_pred_train = best_tree.predict(X_train)
y_pred_test  = best_tree.predict(X_test)

# Metrics
r2_train = r2_score(y_train, y_pred_train)
r2_test  = r2_score(y_test, y_pred_test)

mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test  = mean_absolute_error(y_test, y_pred_test)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test  = np.sqrt(mean_squared_error(y_test, y_pred_test))

print(f"Train R2: {r2_train:.4f}, Test R2: {r2_test:.4f}")
print(f"Train MAE: {mae_train:.2f}, Test MAE: {mae_test:.2f}")
print(f"Train RMSE: {rmse_train:.2f}, Test RMSE: {rmse_test:.2f}")


Cross-validated evaluation

In [ ]:
eval_results_tree = cross_validate(
    best_tree,
    X_train,
    y_train,
    cv=3,
    scoring=["r2", "neg_mean_absolute_error"],
    return_train_score=True
)

test_mae_mean = -eval_results_tree["test_neg_mean_absolute_error"].mean()
test_mae_std  = eval_results_tree["test_neg_mean_absolute_error"].std()
train_mae_mean = -eval_results_tree["train_neg_mean_absolute_error"].mean()
train_mae_std  = eval_results_tree["train_neg_mean_absolute_error"].std()

print(f"CV Test MAE: {test_mae_mean:.2f} ± {test_mae_std:.2f}")
print(f"CV Train MAE: {train_mae_mean:.2f} ± {train_mae_std:.2f}")


Check feature importance

In [ ]:
feature_names = best_tree["preprocessor"].get_feature_names_out()
importances = best_tree["regr"].feature_importances_

tree_importance = pd.Series(importances, index=feature_names).sort_values(ascending=False)
tree_importance.head(15)


# 4 Model Evaluation and Analysis


## 4.1 Coarse-Grained Evaluation/Analysis

collecting all the metrics

In [ ]:
final_results = []

def collect_results(name, model, X_train, y_train, X_test, y_test, cv):
    # Predictions
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    # Metrics
    result = {
        "Model": name,
        "R2 Test": r2_score(y_test, y_pred_test),
        "R2 Train": r2_score(y_train, y_pred_train),
        "MAE Test": mean_absolute_error(y_test, y_pred_test),
        "MAE Train": mean_absolute_error(y_train, y_pred_train),
        "RMSE Test": np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "RMSE Train": np.sqrt(mean_squared_error(y_train, y_pred_train))
    }

    # Cross-validation
    cv_results = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error"
    )

    result["CV MAE Mean"] = -cv_results["test_score"].mean()
    result["CV MAE Std"] = cv_results["test_score"].std()

    final_results.append(result)


adding models

In [ ]:
# Linear Regression (RFECV)
collect_results(
    "Linear Regression (RFECV)",
    regr_rfe,
    X_train,
    y_train,
    X_test,
    y_test,
    cv=5
)


# Decision Tree (Tuned)
collect_results(
    "Decision Tree (Tuned)",
    best_tree,
    X_train,
    y_train,
    X_test,
    y_test,
    cv=3
)

kNN evaluated separately

In [ ]:
# kNN evaluation (separate)

y_pred_train_knn = knn_reg.predict(X_train_knn)
y_pred_test_knn  = knn_reg.predict(X_test_proc)

knn_results = {
    "Model": "kNN Regression (Subsampled)",
    "R2 Test": r2_score(y_test, y_pred_test_knn),
    "R2 Train": r2_score(y_train_knn, y_pred_train_knn),
    "MAE Test": mean_absolute_error(y_test, y_pred_test_knn),
    "MAE Train": mean_absolute_error(y_train_knn, y_pred_train_knn),
    "RMSE Test": np.sqrt(mean_squared_error(y_test, y_pred_test_knn)),
    "RMSE Train": np.sqrt(mean_squared_error(y_train_knn, y_pred_train_knn))
}

cv_knn = cross_validate(
    knn_reg,
    X_train_knn,
    y_train_knn,
    cv=3,
    scoring="neg_mean_absolute_error"
)

knn_results["CV MAE Mean"] = -cv_knn["test_score"].mean()
knn_results["CV MAE Std"]  = cv_knn["test_score"].std()

final_results.append(knn_results)


In [ ]:

# Convert collected results (including kNN) into a DataFrame
final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.round({
    "R2 Test": 3,
    "R2 Train": 3,
    "MAE Test": 2,
    "MAE Train": 2,
    "RMSE Test": 2,
    "RMSE Train": 2,
    "CV MAE Mean": 2,
    "CV MAE Std": 2
})

# Sorts by MAE Test for best model first
final_results_df = final_results_df.sort_values(by="MAE Test", ascending=True).reset_index(drop=True)



In [ ]:
final_results_df

In [ ]:
plot_df = final_results_df.copy()

short_map = {
    "Linear Regression (Baseline)": "LR (Baseline)",
    "Linear Regression (SelectKBest=10)": "LR (SelectKBest)",
    "Linear Regression (RFECV)": "LR (RFECV)",
    "kNN Regression (Subsampled)": "kNN (Subsampled)",
    "Decision Tree (Tuned)": "Tree (Tuned)",
    "Decision Tree (Baseline)": "Tree (Baseline)"
}
plot_df["Model Short"] = plot_df["Model"].replace(short_map)

plot_df = plot_df.sort_values(by="MAE Test", ascending=True)

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

# Left: Test R²
sns.barplot(data=plot_df, x="Model Short", y="R2 Test", ax=axes[0], color="#4C78A8")
axes[0].set_title("Test R² by Model", fontsize=12)
axes[0].set_xlabel("")
axes[0].set_ylabel("R² (Test)")
axes[0].tick_params(axis='x', rotation=20)

for p in axes[0].patches:
    v = p.get_height()
    axes[0].annotate(f"{v:.3f}", (p.get_x() + p.get_width()/2, v),
                     ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

# Right: Test MAE (£)
sns.barplot(data=plot_df, x="Model Short", y="MAE Test", ax=axes[1], color="#59A14F")
axes[1].set_title("Test MAE by Model", fontsize=12)
axes[1].set_xlabel("")
axes[1].set_ylabel("MAE (Test, £)")
axes[1].tick_params(axis='x', rotation=20)

for p in axes[1].patches:
    v = p.get_height()
    axes[1].annotate(f"£{v:,.0f}", (p.get_x() + p.get_width()/2, v),
                     ha='center', va='bottom', fontsize=9, xytext=(0, 2), textcoords='offset points')

fig.suptitle("Model Comparison (Test R² and Test MAE)", fontsize=14)
plt.tight_layout()
plt.show()
#plt.savefig("model comparison.png")


##4.2 Feature Importance

In [ ]:
# Gets feature names and importances
feature_names = best_tree["preprocessor"].get_feature_names_out()
importances = best_tree["regr"].feature_importances_

importance_df = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values(by="Importance", ascending=False)
)

# Plots top 9 features
top_n = 9
plt.figure(figsize=(10, 6))
plt.barh(
    importance_df.head(top_n)["Feature"][::-1],
    importance_df.head(top_n)["Importance"][::-1]
)
plt.xlabel("Relative Importance")
plt.title("Top 9 Feature Importances – Tuned Decision Tree")
plt.tight_layout()
#plt.savefig("features.png")
plt.show()



Cumulative Importance

In [ ]:
importance_df["Cumulative_Importance"] = importance_df["Importance"].cumsum()

plt.figure(figsize=(8, 5))
plt.plot(
    range(1, len(importance_df) + 1),
    importance_df["Cumulative_Importance"],
    marker="o"
)
plt.axhline(0.95, linestyle="--")
plt.xlabel("Number of Features")
plt.ylabel("Cumulative Importance")
plt.title("Cumulative Feature Importance – Decision Tree")
plt.tight_layout()
#plt.savefig("cumuative.png")
plt.show()


##4.3 Fine-Grained Evaluation

Residuals vs Predicted Price

In [ ]:
# Predictions
y_pred_test = best_tree.predict(X_test)

# Residuals
residuals = y_test - y_pred_test


sns.set_style("whitegrid")
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_pred_test, y=residuals, alpha=0.25, s=15)
sns.regplot(x=y_pred_test, y=residuals, scatter=False, color="#d62728", lowess=True)
plt.axhline(0, linestyle="--", color="#333")
plt.xlabel("Predicted Price"); plt.ylabel("Residual (Actual − Predicted)")
plt.title("Residuals vs Predicted Price – Decision Tree (LOWESS trend)")
plt.tight_layout()
plt.show()


Residual Distribution

In [ ]:
lo, hi = np.percentile(residuals, [0.5, 99.5])
lim = max(abs(lo), abs(hi))

bin_width = 500
edges = np.arange(-lim, lim + bin_width, bin_width)


mu = float(np.mean(residuals))
sigma = float(np.std(residuals, ddof=0))

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=edges, range=(-lim, lim), color="#4C78A8", alpha=0.9)

plt.axvline(0, color="black", ls="--", lw=1, label="Zero error")
plt.axvline(mu - sigma, color="#888", ls=":", lw=1)
plt.axvline(mu + sigma, color="#888", ls=":", lw=1)

plt.xlim(-lim, lim)
plt.xlabel("Residual (£)")
plt.ylabel("Frequency")

plt.title(f"Distribution of Prediction Errors – Decision Tree\n"
          f"Mean={mu:,.0f}, SD={sigma:,.0f}  (Bin width = £{bin_width:,})")

plt.legend(frameon=False, loc="upper right")

plt.tight_layout()
plt.show()

Absolute Error vs Vehicle Age

In [ ]:
abs_error = np.abs(residuals)

plt.figure(figsize=(8, 6))
plt.scatter(
    X_test["vehicle_age"],
    abs_error,
    alpha=0.3
)
plt.xlabel("Vehicle Age")
plt.ylabel("Absolute Error")
plt.title("Absolute Prediction Error vs Vehicle Age")
plt.tight_layout()
plt.show()


Absolute Error by Price Band

In [ ]:
# Builds evaluation DataFrame for tuned Decision Tree
error_df = pd.DataFrame({
    "Actual Price": y_test,
    "Predicted Price": best_tree.predict(X_test)
})

# Absolute error
error_df["Absolute Error"] = np.abs(
    error_df["Actual Price"] - error_df["Predicted Price"]
)

# Creates price segments (quintiles)
error_df["Price Segment"] = pd.qcut(
    error_df["Actual Price"],
    q=5
)

# Cleans segment labels
error_df["Price Segment"] = error_df["Price Segment"].astype(str)
error_df["Price Segment"] = error_df["Price Segment"].str.replace(r"\.0", "", regex=True)

# Orders segments from low to high
segment_order = sorted(
    error_df["Price Segment"].unique(),
    key=lambda x: float(x.split(",")[0][1:])
)

# Plots
plt.figure(figsize=(10, 6))

sns.violinplot(
    data=error_df,
    x="Price Segment",
    y="Absolute Error",
    order=segment_order,
    inner="quartile",
    cut=0
)

plt.title("Prediction Error Distribution Across Price Segments")
plt.xlabel("Price Segment (£)")
plt.ylabel("Absolute Prediction Error (£)")


upper = np.percentile(error_df["Absolute Error"], 99)
plt.ylim(0, upper)


plt.tight_layout()
plt.show()


Final Sanity Check Plot

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred_test, alpha=0.3)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle="--"
)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Predicted vs Actual Prices – Decision Tree")
plt.tight_layout()
plt.show()
